# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p), describing second primary colorectal cancer (CRC) survivors, using the `mlcroissant` library via its Croissant schema.

### Dataset Source
The dataset's Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using [`mlcroissant`](https://github.com/mlcommons/croissant).

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print("Description:")
print(metadata.description)
print("\nIdentifier:", metadata.identifier)
print("License:", metadata.license)
print("Version:", metadata.version)
print("Published:", metadata.datePublished)
print("Personal Sensitive Information:", getattr(metadata, 'personalSensitiveInformation', None))

## 2. Data Overview
List available record sets and their fields. All entities are referenced by their `@id` as per the Croissant schema. This overview helps select the appropriate record set(s) to analyze.

In [ ]:
# Get all record sets by @id (using Croissant API)
record_sets = dataset.record_sets

# Print the @id and field structure of each record set
print("Available record sets (@id):\n")
recset_ids = []
for recset in record_sets:
    print(f"- {recset['@id']} (name: {recset.get('name', '<unnamed>')})")
    recset_ids.append(recset['@id'])
    if 'field' in recset:
        fields = recset['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("   Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"     • {field['@id']} (name: {field.get('name', '<unnamed>')})")
            else:
                print(f"     • {field}")
    print()
if not recset_ids:
    print("No record sets found in this schema. The dataset might be defined in a minimally structured way.")

## 3. Data Extraction
Extract records from a chosen record set using its `@id`, and load them into a pandas DataFrame for analysis.

In [ ]:
# If no record sets are listed, find the main tabular data record set by consulting the documentation
# For demonstration, let's list all dataframes for all discovered record sets
dataframes = {}

for recset_id in recset_ids:
    print(f"Loading records for record set: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Shape: {df.shape}, Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Failed to load records from {recset_id}: {e}")

# If only one or a main record set is present, we select it for downstream operations
if len(dataframes) == 1:
    chosen_recset_id = list(dataframes.keys())[0]
else:
    # You may manually select a record set or default to the first one
    chosen_recset_id = recset_ids[0] if recset_ids else None

print("\nChosen record set for EDA:", chosen_recset_id)
if chosen_recset_id:
    print("Available columns:", dataframes[chosen_recset_id].columns.tolist())
    display(dataframes[chosen_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data cleaning and transformation to one numerical field and group the data by a categorical attribute. Please update the field `@id`s used below according to the Data Overview above.

In [ ]:
import numpy as np

# Choose a representative numeric field and group field by their @id
# Please replace these with actual @id values if needed (see Section 2 output)

numeric_field_id = None
group_field_id = None
df = dataframes[chosen_recset_id]

# Suggest a numeric and a group field automatically if not set
if numeric_field_id is None:
    # Heuristic: Try to find integer/float-like columns
    num_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if num_candidates:
        numeric_field_id = num_candidates[0]
if group_field_id is None:
    # Heuristic: Try to find a likely categorical column
    cat_candidates = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < df.shape[0]/2]
    if cat_candidates:
        group_field_id = cat_candidates[0]

print(f"Numeric field selected (@id/column): {numeric_field_id}")
print(f"Group field selected (@id/column): {group_field_id}")

if numeric_field_id:
    # Remove missing values for demonstration
    subdf = df[df[numeric_field_id].notnull()].copy()

    # Example filter: Greater than a threshold (use median if data is not 0-centered)
    threshold = subdf[numeric_field_id].median()
    filtered_df = subdf[subdf[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field (z-score)
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm]].head())

    # Group by group field if provided
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name='mean_' + numeric_field_id)
        print(f"Grouped (mean) by {group_field_id}:")
        display(grouped_df.reset_index().head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Generate simple plots to visualize the distribution of a numeric field and its relationship to a group variable. You may modify the field names below to fit your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- This notebook demonstrated how to load, inspect, and analyze a multi-field, multi-record set tabular biomedical dataset described by a Croissant schema, using the `mlcroissant` Python package.
- We referenced all entities using their `@id`, as found in the schema and provided programmatically by the library.
- Data extraction, cleaning, transformation, and basic visualization can be easily adapted to any tabular dataset described in the Croissant format.
- For more advanced use, explore additional record sets, expand the EDA, or join across sets using the `@id` of fields or entities of interest.